In [2]:
import numpy as np
from numba import njit

In [58]:
import numpy as np
from numba import njit

@njit
def numba_nansum(arr, axis=0):
    return np.where(np.isnan(arr), 0, arr).sum(axis=axis)

@njit
def get_pt_utility(lamda, alpha, outcomes):
    num_outcomes_per_option = outcomes.shape[1] // 2  # Must be even
    outcome_prob = 1 / num_outcomes_per_option
    values = np.where(outcomes >= 0, outcomes**alpha, -lamda * np.abs(outcomes)**alpha)
    # Compute utilities for option A and B using slicing
    utility_a = outcome_prob * numba_nansum(values[:, :num_outcomes_per_option], axis=1)
    utility_b = outcome_prob * numba_nansum(values[:, num_outcomes_per_option:], axis=1)
    return np.stack((utility_a, utility_b), axis=1)

@njit
def get_choices(utilities, tau):
    num_choices = utilities.shape[0]
    choices = np.zeros(num_choices, dtype=np.int64)
    # Compute softmax probabilities
    e_x = np.exp(utilities * tau)
    probs = e_x / np.sum(e_x, axis=1)[:, None]
    rand_vals = np.random.random(num_choices)
    # Determine choices using searchsorted
    for i in range(num_choices):
        cum_prob = np.cumsum(probs[i])
        choices[i] = np.searchsorted(cum_prob, rand_vals[i], side="right")
    return choices

In [59]:
outcomes = np.array([
    [20, 5, -2, 3, -2, 1],  # Sample 1
    [10, -5, -2, 3, -2, 1],  # Sample 1
    [10, -5, -2, 3, -2, 1],  # Sample 1
    [1000, -5, -2, 3, -2, 1],  # Sample 1
    [10, -5, -2, 3, -2, 1],  # Sample 1
    [10, -5, -2, 3, 100, 1] # Sample 1
])

lamda = 1
alpha = 0.5

In [72]:
%%time
utilities = get_pt_utility(lamda, alpha, outcomes)
choices = get_choices(utilities, np.array([[20.0, 20.0, 20.0, 20.0, 20.0, 20.0]]).reshape((6, 1)))
choices

CPU times: user 45 μs, sys: 0 ns, total: 45 μs
Wall time: 46.7 μs


array([0, 1, 1, 0, 1, 1])

In [11]:
np.array([[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]]).reshape((6, 1))

array([[1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.]])

In [477]:
@njit
def get_choice(x, tau):
    """Computes probabilistic choices for a batch of inputs with manual cumulative sum."""
    batch_size = x.shape[0]

    # Compute softmax probabilities
    e_x = np.exp(x * tau)
    
    # Normalize probabilities by summing along the correct axis
    probs = e_x / np.sum(e_x, axis=1)[:, None]  # Normalize along rows (batch axis)

    # Manually compute the cumulative sum
    cum_probs = np.cumsum(probs, axis=1)

    # Generate random numbers for each batch sample
    # rand_vals = np.random.random(batch_size)

    # Use np.digitize to determine the indices for each random value
    # choices = np.digitize(rand_vals, cum_probs)

    # return choices

# Example usage
batch_size = 5
x = np.array([
    [1.0, 2.0],
    [0.5, 1.5],
    [2.0, 1.0],
    [1.2, 1.8],
    [0.7, 1.3]
])
tau = 1.0

# Get choices for the batch
choices = get_choice(x, tau)
print("Choices:", choices)  

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1mNo implementation of function Function(<function cumsum at 0x104b09300>) found for signature:
 
 >>> cumsum(array(float64, 2d, C), axis=Literal[int](1))
 
There are 2 candidate implementations:
[1m      - Of which 2 did not match due to:
      Overload in function 'array_cumsum': File: numba/np/arraymath.py: Line 375.
        With argument(s): '(array(float64, 2d, C), axis=int64)':[0m
[1m       Rejected as the implementation raised a specific error:
         TypingError: [1mgot an unexpected keyword argument 'axis'[0m[0m
  raised from /Users/lschumacher/miniconda3/envs/bf_dev/lib/python3.11/site-packages/numba/core/typing/templates.py:783
[0m
[0m[1mDuring: resolving callee type: Function(<function cumsum at 0x104b09300>)[0m
[0m[1mDuring: typing of call at /var/folders/9q/trrmd4jn4bx40rltjz_fmx4h0000gp/T/ipykernel_35974/432010770.py (13)
[0m
[1m
File "../../../../../var/folders/9q/trrmd4jn4